# PW04 · Attack Merge And Metrics

用途：在 Colab 会话中执行 PW04 attack merge、formal overlay materialize 与 attack metrics 导出。PW04 只消费 PW02 finalize 与 thresholds，以及全部已完成的 PW03 attack shards。

边界：这是纯 merge / materialize / aggregate 阶段，不重新执行 embed、detect、calibrate，也不重新生成 clean source 或 attacked image。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW04_Attack_Merge_And_Metrics"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "pw04_merge_attack_event_shards.py"
LOCAL_HF_HOME = REPO_ROOT / "huggingface_cache"
LOCAL_HF_HUB_CACHE = LOCAL_HF_HOME / "hub"
LOCAL_TRANSFORMERS_CACHE = LOCAL_HF_HOME / "transformers"

FAMILY_ID = "paper_eval_family_demo"
FORCE_RERUN = False
ENABLE_TAIL_ESTIMATION = False

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

MODEL_CACHE_LAYOUT = resolve_notebook_model_cache_layout(DRIVE_MOUNT_ROOT, REPO_ROOT, create_directories=True)
LOCAL_HF_HOME = MODEL_CACHE_LAYOUT["local_hf_home"]
LOCAL_HF_HUB_CACHE = MODEL_CACHE_LAYOUT["local_hf_hub_cache"]
LOCAL_TRANSFORMERS_CACHE = MODEL_CACHE_LAYOUT["local_transformers_cache"]
os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "local_hf_home": str(LOCAL_HF_HOME),
        "local_hf_hub_cache": str(LOCAL_HF_HUB_CACHE),
        "local_transformers_cache": str(LOCAL_TRANSFORMERS_CACHE),
    },
)

In [ ]:
FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
PW02_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw02_summary.json"
FINALIZE_MANIFEST_PATH = FAMILY_ROOT / "exports" / "pw02" / "paper_source_finalize_manifest.json"
CONTENT_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "content" / "thresholds.json"
ATTESTATION_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "attestation" / "thresholds.json"
ATTACK_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "attack_shard_plan.json"
ATTACK_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "attack_event_grid.jsonl"
PW04_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw04_summary.json"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW04 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("PW02_SUMMARY_PATH 存在", PW02_SUMMARY_PATH.exists(), str(PW02_SUMMARY_PATH))
record_precheck("FINALIZE_MANIFEST_PATH 存在", FINALIZE_MANIFEST_PATH.exists(), str(FINALIZE_MANIFEST_PATH))
record_precheck("content thresholds export 存在", CONTENT_THRESHOLD_EXPORT_PATH.exists(), str(CONTENT_THRESHOLD_EXPORT_PATH))
record_precheck("attestation thresholds export 存在", ATTESTATION_THRESHOLD_EXPORT_PATH.exists(), str(ATTESTATION_THRESHOLD_EXPORT_PATH))
record_precheck("ATTACK_SHARD_PLAN_PATH 存在", ATTACK_SHARD_PLAN_PATH.exists(), str(ATTACK_SHARD_PLAN_PATH))
record_precheck("ATTACK_EVENT_GRID_PATH 存在", ATTACK_EVENT_GRID_PATH.exists(), str(ATTACK_EVENT_GRID_PATH))

ATTACK_SHARD_PLAN = {}
EXPECTED_ATTACK_EVENT_COUNT = None
DISCOVERED_ATTACK_EVENT_COUNT = 0
PLANNED_SHARD_RESULTS = []
if ATTACK_SHARD_PLAN_PATH.exists():
    ATTACK_SHARD_PLAN = json.loads(ATTACK_SHARD_PLAN_PATH.read_text(encoding="utf-8"))
    EXPECTED_ATTACK_EVENT_COUNT = ATTACK_SHARD_PLAN.get("attack_event_count")
    for shard_row in ATTACK_SHARD_PLAN.get("shards", []):
        shard_index = int(shard_row["attack_shard_index"])
        shard_manifest_path = FAMILY_ROOT / "attack_shards" / f"shard_{shard_index:04d}" / "shard_manifest.json"
        shard_exists = shard_manifest_path.exists()
        shard_status = "<absent>"
        shard_event_count = 0
        if shard_exists:
            shard_manifest = json.loads(shard_manifest_path.read_text(encoding="utf-8"))
            shard_status = str(shard_manifest.get("status", "<absent>"))
            if isinstance(shard_manifest.get("event_count"), int):
                shard_event_count = int(shard_manifest["event_count"])
            else:
                shard_event_count = len(shard_manifest.get("events", []))
            if shard_status == "completed":
                DISCOVERED_ATTACK_EVENT_COUNT += shard_event_count
        PLANNED_SHARD_RESULTS.append({
            "attack_shard_index": shard_index,
            "path": str(shard_manifest_path),
            "exists": shard_exists,
            "status": shard_status,
            "event_count": shard_event_count,
        })

record_precheck("所有计划内 PW03 shard manifest 存在且 completed", all(item["exists"] and item["status"] == "completed" for item in PLANNED_SHARD_RESULTS), json.dumps(PLANNED_SHARD_RESULTS, ensure_ascii=False, indent=2))
record_precheck("expected_attack_event_count == discovered_attack_event_count", isinstance(EXPECTED_ATTACK_EVENT_COUNT, int) and EXPECTED_ATTACK_EVENT_COUNT == DISCOVERED_ATTACK_EVENT_COUNT, f"expected_attack_event_count={EXPECTED_ATTACK_EVENT_COUNT}, discovered_attack_event_count={DISCOVERED_ATTACK_EVENT_COUNT}")

print_json("pw04_precheck", {"items": PRECHECK_RESULTS})
hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW04 precheck failed: {hard_fail}")

In [ ]:
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
]
if FORCE_RERUN:
    COMMAND.append("--force-rerun")
if ENABLE_TAIL_ESTIMATION:
    COMMAND.append("--enable-tail-estimation")

PW04_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=build_repo_import_subprocess_env(repo_root=REPO_ROOT),
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if PW04_RESULT.returncode != 0:
    raise RuntimeError(
        "PW04 failed: "
        f"return_code={PW04_RESULT.returncode} stdout={PW04_RESULT.stdout} stderr={PW04_RESULT.stderr}"
    )
if not PW04_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"PW04 summary file missing after successful execution: {PW04_SUMMARY_PATH}")

PW04_SUMMARY = json.loads(PW04_SUMMARY_PATH.read_text(encoding="utf-8"))
print_json("pw04_summary", PW04_SUMMARY)

In [ ]:
FORMAL_ATTACK_FINAL_DECISION_METRICS_PATH = FAMILY_ROOT / "exports" / "pw04" / "formal_attack_final_decision_metrics.json"
FORMAL_ATTACK_ATTESTATION_METRICS_PATH = FAMILY_ROOT / "exports" / "pw04" / "formal_attack_attestation_metrics.json"
DERIVED_ATTACK_UNION_METRICS_PATH = FAMILY_ROOT / "exports" / "pw04" / "derived_attack_union_metrics.json"
CLEAN_ATTACK_OVERVIEW_PATH = FAMILY_ROOT / "exports" / "pw04" / "clean_attack_overview.json"

PAPER_SCOPE_REGISTRY_PATH = Path(str(PW04_SUMMARY["paper_scope_registry_path"]))
CANONICAL_METRICS_PATHS = dict(PW04_SUMMARY.get("canonical_metrics_paths", {}))
PAPER_TABLE_PATHS = dict(PW04_SUMMARY.get("paper_tables_paths", {}))
PAPER_FIGURE_PATHS = dict(PW04_SUMMARY.get("paper_figures_paths", {}))
TAIL_ESTIMATION_PATHS = dict(PW04_SUMMARY.get("tail_estimation_paths", {}))
BOOTSTRAP_CONFIDENCE_INTERVALS_PATH = Path(str(PW04_SUMMARY["bootstrap_confidence_intervals_path"]))
BOOTSTRAP_CONFIDENCE_INTERVALS_CSV_PATH = Path(str(PW04_SUMMARY["bootstrap_confidence_intervals_csv_path"]))

PW04_RESULT_SUMMARY = {
    "summary": json.loads(PW04_SUMMARY_PATH.read_text(encoding="utf-8")),
    "formal_attack_final_decision_metrics": json.loads(FORMAL_ATTACK_FINAL_DECISION_METRICS_PATH.read_text(encoding="utf-8")),
    "formal_attack_attestation_metrics": json.loads(FORMAL_ATTACK_ATTESTATION_METRICS_PATH.read_text(encoding="utf-8")),
    "derived_attack_union_metrics": json.loads(DERIVED_ATTACK_UNION_METRICS_PATH.read_text(encoding="utf-8")),
    "clean_attack_overview": json.loads(CLEAN_ATTACK_OVERVIEW_PATH.read_text(encoding="utf-8")),
    "paper_metric_registry": json.loads(PAPER_SCOPE_REGISTRY_PATH.read_text(encoding="utf-8")),
    "content_chain_metrics": json.loads(Path(str(CANONICAL_METRICS_PATHS["content_chain"])).read_text(encoding="utf-8")),
    "event_attestation_metrics": json.loads(Path(str(CANONICAL_METRICS_PATHS["event_attestation"])).read_text(encoding="utf-8")),
    "system_final_metrics": json.loads(Path(str(CANONICAL_METRICS_PATHS["system_final"])).read_text(encoding="utf-8")),
    "bootstrap_confidence_intervals": json.loads(BOOTSTRAP_CONFIDENCE_INTERVALS_PATH.read_text(encoding="utf-8")),
    "bootstrap_confidence_intervals_csv_path": str(BOOTSTRAP_CONFIDENCE_INTERVALS_CSV_PATH),
    "paper_tables_paths": PAPER_TABLE_PATHS,
    "paper_figures_paths": {
        key: {"path": value, "exists": Path(str(value)).exists()}
        for key, value in PAPER_FIGURE_PATHS.items()
    },
    "tail_estimation_paths": TAIL_ESTIMATION_PATHS,
    "estimated_tail_fpr_1e4": json.loads(Path(str(TAIL_ESTIMATION_PATHS["estimated_tail_fpr_1e4_path"])).read_text(encoding="utf-8")),
    "estimated_tail_fpr_1e5": json.loads(Path(str(TAIL_ESTIMATION_PATHS["estimated_tail_fpr_1e5_path"])).read_text(encoding="utf-8")),
    "tail_fit_diagnostics": json.loads(Path(str(TAIL_ESTIMATION_PATHS["tail_fit_diagnostics_path"])).read_text(encoding="utf-8")),
    "tail_fit_stability_summary": json.loads(Path(str(TAIL_ESTIMATION_PATHS["tail_fit_stability_summary_path"])).read_text(encoding="utf-8")),
}
print_json("pw04_result_summary", PW04_RESULT_SUMMARY)